# 03 — Evaluation, Confusion Matrix, and Visualization

**Purpose:** evaluate the saved classifier using a labeled test set, generate a confusion matrix, print a classification report, and create patch-grid visualization outputs.

A confusion matrix requires a labeled test folder such as:

```text
split_dataset/test/Adults/*.jpg
split_dataset/test/Children/*.jpg
split_dataset/test/Seniors/*.jpg
```

### Step 1 — Save evaluation script

In [ ]:
import os
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

from train_model import IMAGE_SIZE, BATCH_SIZE, NUM_CLASSES, DEVICE, TRAIN_DIR, MODEL_SAVE, val_transforms

# Use a labeled test folder for quantitative evaluation.
# Expected folder structure:
#   split_dataset/test/adults/*.jpg
#   split_dataset/test/children/*.jpg
#   split_dataset/test/seniors/*.jpg
LABELED_TEST_DIR = "split_dataset/test"


def build_eval_model(model_path=MODEL_SAVE):
    # Build the same EfficientNet-B0 architecture and load saved weights.
    model = models.efficientnet_b0(weights=None)
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.4),
        nn.Linear(in_features, 128),
        nn.ReLU(),
        nn.Dropout(p=0.3),
        nn.Linear(128, NUM_CLASSES),
    )
    model.load_state_dict(torch.load(model_path, map_location=DEVICE))
    model.to(DEVICE)
    model.eval()
    return model


def build_labeled_test_loader(test_dir=LABELED_TEST_DIR, train_dir=TRAIN_DIR):
    #Loads a labeled test set and keeps the label order consistent with the training folder.
    #This avoids class-index mismatch between train and test folders.
    if not os.path.isdir(test_dir):
        raise FileNotFoundError(
            f"Could not find labeled test folder: {test_dir}. "
            # For confusion matrix, test_dir must contain class subfolders
        )

    train_dataset = datasets.ImageFolder(train_dir)
    train_class_to_idx = train_dataset.class_to_idx
    class_names = train_dataset.classes

    raw_test = datasets.ImageFolder(test_dir, transform=val_transforms)
    test_class_to_idx = raw_test.class_to_idx

    remap = {}
    for class_name, test_idx in test_class_to_idx.items():
        if class_name not in train_class_to_idx:
            raise ValueError(
                f"Class '{class_name}' exists in test set but not in train set. "
                f"Train classes: {list(train_class_to_idx.keys())}"
            )
        remap[test_idx] = train_class_to_idx[class_name]

    raw_test.samples = [(path, remap[label]) for path, label in raw_test.samples]
    raw_test.targets = [remap[label] for label in raw_test.targets]
    raw_test.classes = class_names
    raw_test.class_to_idx = train_class_to_idx

    test_loader = DataLoader(raw_test, batch_size=BATCH_SIZE, shuffle=False)
    return test_loader, raw_test, class_names


def collect_predictions(model, test_loader):
    all_preds = []
    all_labels = []
    all_confidences = []

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(DEVICE)
            outputs = model(inputs)
            probs = torch.softmax(outputs, dim=1)
            confidences, preds = torch.max(probs, 1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
            all_confidences.extend(confidences.cpu().numpy())

    return np.array(all_preds), np.array(all_labels), np.array(all_confidences)


def analyze_individual_files(test_dir=LABELED_TEST_DIR, model_path=MODEL_SAVE):
    test_loader, test_dataset, class_names = build_labeled_test_loader(test_dir=test_dir)
    model = build_eval_model(model_path=model_path)

    print(f"Analyzing {len(test_dataset)} labeled test images...\n")

    all_preds, all_labels, all_confidences = collect_predictions(model, test_loader)

    print(f"{'FILENAME':<40} | {'TRUE CLASS':<12} | {'PREDICTION':<12} | {'CONFIDENCE'}")
    print("-" * 95)

    wrong_count = 0
    for i, (img_path, _) in enumerate(test_dataset.samples):
        filename = os.path.basename(img_path)
        true_class = class_names[all_labels[i]]
        pred_class = class_names[all_preds[i]]
        confidence = all_confidences[i]
        is_correct = all_labels[i] == all_preds[i]
        status = "✓" if is_correct else "✗"
        wrong_count += int(not is_correct)
        print(f"{filename:<40} | {true_class:<12} | {pred_class:<12} | {confidence:.2%} {status}")

    print("\n" + "=" * 45)
    print(f"TOTAL WRONG: {wrong_count} / {len(test_dataset)}")
    print(f"OVERALL ACCURACY: {accuracy_score(all_labels, all_preds):.2%}")
    print("=" * 45)

    if wrong_count > 0:
        print("\nMisclassified files:")
        for i, (img_path, _) in enumerate(test_dataset.samples):
            if all_labels[i] != all_preds[i]:
                filename = os.path.basename(img_path)
                print(
                    f" - {filename}: was {class_names[all_labels[i]]}, "
                    f"predicted {class_names[all_preds[i]]} "
                    f"({all_confidences[i]:.2%})"
                )

    return all_preds, all_labels, all_confidences, class_names


def generate_confusion_matrix(test_dir=LABELED_TEST_DIR, model_path=MODEL_SAVE, save_path="confusion_matrix.png"):
    test_loader, test_dataset, class_names = build_labeled_test_loader(test_dir=test_dir)
    model = build_eval_model(model_path=model_path)

    print("Gathering predictions for confusion matrix...")
    all_preds, all_labels, _ = collect_predictions(model, test_loader)

    cm = confusion_matrix(all_labels, all_preds)

    plt.figure(figsize=(10, 8))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=class_names,
        yticklabels=class_names,
    )
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title("Confusion Matrix: Age Group Classifier")
    plt.tight_layout()
    plt.savefig(save_path, dpi=200)
    plt.show()

    print(f"\nConfusion matrix saved to: {save_path}")
    print("\nDetailed Classification Report:")
    print(classification_report(all_labels, all_preds, target_names=class_names, zero_division=0))

    return cm, all_preds, all_labels, class_names


if __name__ == "__main__":
    analyze_individual_files()
    generate_confusion_matrix()


### Step 2 — Save visualization script

It runs on flattened images inside `dataset/test/` and saves results in `results_updated/`.

In [ ]:
import os
import torch
import torch.nn as nn
import numpy as np
from torchvision import transforms, models
from PIL import Image, ImageDraw, ImageFont

# Main paths and settings
TEST_DIR        = "dataset/test"
RESULTS_DIR     = "results_updated"
MODEL_PATH      = "age_classifier.pth"
IMAGE_SIZE      = 224
DEVICE          = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Grid Settings: Divide image into a 4x4 grid (16 squares total)
GRID_COLS       = 4
GRID_ROWS       = 4

# Transparency of the color overlay (0=invisible, 255=solid)
OVERLAY_ALPHA   = 110

CLASS_NAMES     = ["adults", "children", "seniors"]

# Colors per class: (R, G, B)
CLASS_COLORS = {
    "adults":   (66,  133, 244),   # Blue
    "children": (52,  168,  83),   # Green
    "seniors":  (234,  67,  53),   # Red
}

# Image preprocessing for prediction
infer_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

# Loading the model
def load_model(model_path):
    model = models.efficientnet_b0(weights=None)
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.4),
        nn.Linear(in_features, 128),
        nn.ReLU(),
        nn.Dropout(p=0.3),
        nn.Linear(128, 3),
    )
    model.load_state_dict(torch.load(model_path, map_location=DEVICE))
    model.to(DEVICE)
    model.eval()
    print(f" Model loaded from: {model_path}")
    return model

# Classifying a single patch
def classify_patch(model, patch_img):
    # Takes a small square of an image and predicts the age group for just that square
    tensor = infer_transform(patch_img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        logits = model(tensor)
        probs  = torch.softmax(logits, dim=1).squeeze().cpu().numpy()
    pred_idx   = int(np.argmax(probs))
    return CLASS_NAMES[pred_idx], float(probs[pred_idx]), probs


def process_image(model, img_path, save_path):
    img = Image.open(img_path).convert("RGB")
    W, H = img.size

    # Split image into equal patches
    patch_w = W // GRID_COLS
    patch_h = H // GRID_ROWS

    # Create a transparent layer to draw the colors on
    overlay = Image.new("RGBA", (W, H), (0, 0, 0, 0))
    draw    = ImageDraw.Draw(overlay)

    class_vote_counts = {cls: 0 for cls in CLASS_NAMES}

    try:
        patch_font = ImageFont.truetype(
            "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 13)
    except:
        patch_font = ImageFont.load_default()

    # Go through every square in the grid
    for row in range(GRID_ROWS):
        for col in range(GRID_COLS):
            x0 = col * patch_w
            y0 = row * patch_h
            x1 = x0 + patch_w if col < GRID_COLS - 1 else W
            y1 = y0 + patch_h if row < GRID_ROWS - 1 else H

            patch        = img.crop((x0, y0, x1, y1))
            cls, conf, _ = classify_patch(model, patch)
            class_vote_counts[cls] += 1

            # Fill patch with its class color
            color = CLASS_COLORS[cls] + (OVERLAY_ALPHA,)
            draw.rectangle([x0, y0, x1, y1], fill=color)

            label = f"{cls[:3].upper()} {conf:.0%}"
            draw.text((x0 + 4, y0 + 4), label, fill=(255, 255, 255, 230), font=patch_font)
            draw.rectangle([x0, y0, x1 - 1, y1 - 1],
                           outline=(255, 255, 255, 160), width=1)

    # Get the class with the highest number of patch votes
    majority_class = max(class_vote_counts, key=class_vote_counts.get)
    majority_count = class_vote_counts[majority_class]

    # Merge overlay with original image
    img_rgba   = img.convert("RGBA")
    composited = Image.alpha_composite(img_rgba, overlay)
    result_img = composited.convert("RGB")
    result_draw = ImageDraw.Draw(result_img)

    try:
        title_font  = ImageFont.truetype(
            "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 20)
        legend_font = ImageFont.truetype(
            "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 16)
        small_font  = ImageFont.truetype(
            "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", 14)
    except:
        title_font = legend_font = small_font = ImageFont.load_default()

    # Top banner with majority result
    banner_color = CLASS_COLORS[majority_class]
    result_draw.rectangle([0, 0, W, 40], fill=banner_color)
    vote_summary = "  |  ".join(
        [f"{cls.capitalize()}: {class_vote_counts[cls]}" for cls in CLASS_NAMES]
    )
    result_draw.text(
        (10, 9),
        f"MAJORITY: {majority_class.upper()}  ({majority_count}/{GRID_COLS*GRID_ROWS} patches)   [{vote_summary}]",
        fill=(255, 255, 255),
        font=title_font,
    )

    # Bottom legend box
    lx, ly = 10, H - 95
    result_draw.rectangle([lx - 6, ly - 6, lx + 185, ly + 88],
                          fill=(20, 20, 20))
    result_draw.text((lx, ly), "Legend", fill=(255, 255, 255), font=legend_font)
    ly += 24
    for cls, color in CLASS_COLORS.items():
        result_draw.rectangle([lx, ly, lx + 16, ly + 16], fill=color)
        result_draw.text((lx + 22, ly), cls.capitalize(),
                         fill=(255, 255, 255), font=small_font)
        ly += 22

    result_img.save(save_path, quality=95)
    return majority_class, class_vote_counts


def main():
    os.makedirs(RESULTS_DIR, exist_ok=True)

    print(f" Running on: {DEVICE}")
    model = load_model(MODEL_PATH)

    supported   = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
    image_files = sorted([
        f for f in os.listdir(TEST_DIR)
        if f.lower().endswith(supported)
    ])

    if not image_files:
        print(f" No images found in {TEST_DIR}")
        return

    print(f"\n Processing {len(image_files)} images with {GRID_ROWS}×{GRID_COLS} patch grid...\n")
    print(f"{'Image':<50} {'Majority':<12} Patch Votes")
    print("─" * 80)

    for fname in image_files:
        img_path  = os.path.join(TEST_DIR, fname)
        save_name = os.path.splitext(fname)[0] + "_result.jpg"
        save_path = os.path.join(RESULTS_DIR, save_name)

        try:
            majority, votes = process_image(model, img_path, save_path)
            vote_str = "  ".join([f"{c}: {v}" for c, v in votes.items()])
            print(f"{fname:<50} {majority:<12} {vote_str}")
        except Exception as e:
            print(f"{fname:<50} ERROR: {e}")

    print(f"\n All results saved to: {RESULTS_DIR}/")
    print(f" Blue = Adults  | Green = Children  |  Red = Seniors")

if __name__ == "__main__":
    main()
